# Derivação de publish_gap e connect_gap (MitM e Intrusion)

Notebook criado para derivar `publish_gap` e `connect_gap` nos datasets `MitM.csv` e `Intrusion.csv`, evitando data leakage por meio de cálculo após splits estratificados. Também remove `mqtt.proto_len` e `mqtt.ver` e salva versões enriquecidas (`MitM_gap.csv` e `intrusion_gap.csv`).

## 1) Configuração do ambiente e carregamento dos dados

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

# Caminhos de entrada/saída
root = Path("..")
raw_dir = root / "data" / "raw" / "MQTT Under Attack Dataset"
processed_dir = root / "data" / "processed"

mitm_path = raw_dir / "MitM.csv"
intrusion_path = raw_dir / "Intrusion.csv"
mitm_out = processed_dir / "MitM_gap.csv"
intrusion_out = processed_dir / "intrusion_gap.csv"

print("Entradas:", mitm_path, intrusion_path)
print("Saídas:", mitm_out, intrusion_out)


Entradas: ../data/raw/MQTT Under Attack Dataset/MitM.csv ../data/raw/MQTT Under Attack Dataset/Intrusion.csv
Saídas: ../data/processed/MitM_gap.csv ../data/processed/intrusion_gap.csv


In [2]:
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Ordenar por tempo para coerência nas operações de diff
    if "frame.time_epoch" in df.columns:
        df = df.sort_values("frame.time_epoch").reset_index(drop=True)
    return df

mitm_df = load_dataset(mitm_path)
intrusion_df = load_dataset(intrusion_path)

for name, df in {"MitM": mitm_df, "Intrusion": intrusion_df}.items():
    print(f"{name}: shape={df.shape}, cols={len(df.columns)}")
    print(df['type'].value_counts())
    print("-")


/tmp/ipykernel_510446/507520413.py:2: DtypeWarning: Columns (0: mqtt.clientid, 1: mqtt.conack.flags, 2: mqtt.conflags, 3: mqtt.protoname, 4: mqtt.topic) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


MitM: shape=(110668, 67), cols=67
type
normal    106813
mitm        3855
Name: count, dtype: int64
-
Intrusion: shape=(80893, 67), cols=67
type
normal       78995
intrusion     1898
Name: count, dtype: int64
-


## 2) Limpeza inicial e exclusão de colunas

In [3]:
drop_cols = ['mqtt.proto_len', 'mqtt.ver']

def drop_unwanted(df: pd.DataFrame) -> pd.DataFrame:
    cols_present = [c for c in drop_cols if c in df.columns]
    return df.drop(columns=cols_present, errors='ignore')

mitm_df = drop_unwanted(mitm_df)
intrusion_df = drop_unwanted(intrusion_df)

for name, df in {"MitM": mitm_df, "Intrusion": intrusion_df}.items():
    print(f"{name}: colunas após drop = {len(df.columns)}")


MitM: colunas após drop = 65
Intrusion: colunas após drop = 65


## 3) Funções auxiliares para criar publish_gap e connect_gap sem vazamento

In [4]:
def compute_gap_per_partition(df_part: pd.DataFrame, msgtype_value: int, time_col: str = "frame.time_epoch") -> pd.Series:
    if time_col not in df_part.columns:
        raise ValueError(f"Coluna de tempo {time_col} não encontrada")
    series = pd.Series(np.zeros(len(df_part)), index=df_part.index, dtype=float)
    mask = df_part['mqtt.msgtype'] == msgtype_value
    if mask.any():
        ordered = df_part.loc[mask].sort_values(time_col)
        gaps = ordered[time_col].diff().fillna(0)
        series.loc[ordered.index] = gaps.values
    return series


def add_gaps_no_leak(df: pd.DataFrame, test_size: float = 0.3, random_state: int = 42):
    # Split estratificado para evitar vazamento temporal entre splits
    idx = np.arange(len(df))
    train_idx, test_idx = train_test_split(idx, test_size=test_size, random_state=random_state, stratify=df['type'])
    df_train = df.iloc[train_idx].copy().reset_index(drop=True)
    df_test = df.iloc[test_idx].copy().reset_index(drop=True)

    def _add(df_split: pd.DataFrame) -> pd.DataFrame:
        df_split = df_split.sort_values('frame.time_epoch').reset_index(drop=True)
        df_split['publish_gap'] = compute_gap_per_partition(df_split, msgtype_value=3)
        df_split['connect_gap'] = compute_gap_per_partition(df_split, msgtype_value=1)
        return df_split

    df_train = _add(df_train)
    df_test = _add(df_test)

    combined = pd.concat([df_train, df_test], ignore_index=True)
    return combined, df_train, df_test


## 4) Criação das features sem vazamento (MitM)

In [5]:
mitm_with_gaps, mitm_train, mitm_test = add_gaps_no_leak(mitm_df)
print("MitM - publish_gap stats:\n", mitm_with_gaps['publish_gap'].describe())
print("MitM - connect_gap stats:\n", mitm_with_gaps['connect_gap'].describe())


MitM - publish_gap stats:
 count    110668.000000
mean          0.083556
std           4.499078
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         813.081635
Name: publish_gap, dtype: float64
MitM - connect_gap stats:
 count    110668.000000
mean          0.014554
std           3.798768
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        1192.538111
Name: connect_gap, dtype: float64


## 5) Criação das features sem vazamento (Intrusion)

In [6]:
intrusion_with_gaps, intrusion_train, intrusion_test = add_gaps_no_leak(intrusion_df)
print("Intrusion - publish_gap stats:\n", intrusion_with_gaps['publish_gap'].describe())
print("Intrusion - connect_gap stats:\n", intrusion_with_gaps['connect_gap'].describe())


Intrusion - publish_gap stats:
 count    80893.000000
mean         0.123156
std          3.490596
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        561.140922
Name: publish_gap, dtype: float64
Intrusion - connect_gap stats:
 count    80893.000000
mean         0.121671
std          5.955592
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        876.639137
Name: connect_gap, dtype: float64


## 6) Validação básica das novas features

In [7]:
def summarize(df: pd.DataFrame, name: str):
    print(f"\n{name} - publish_gap resumo:")
    print(df['publish_gap'].describe())
    print(f"\n{name} - connect_gap resumo:")
    print(df['connect_gap'].describe())

summarize(mitm_with_gaps, "MitM")
summarize(intrusion_with_gaps, "Intrusion")



MitM - publish_gap resumo:
count    110668.000000
mean          0.083556
std           4.499078
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         813.081635
Name: publish_gap, dtype: float64

MitM - connect_gap resumo:
count    110668.000000
mean          0.014554
std           3.798768
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        1192.538111
Name: connect_gap, dtype: float64

Intrusion - publish_gap resumo:
count    80893.000000
mean         0.123156
std          3.490596
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        561.140922
Name: publish_gap, dtype: float64

Intrusion - connect_gap resumo:
count    80893.000000
mean         0.121671
std          5.955592
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        876.639137
Name: connect_gap, dtype: float64


## 7) Salvar datasets enriquecidos

In [8]:
processed_dir.mkdir(parents=True, exist_ok=True)

mitm_with_gaps.to_csv(mitm_out, index=False)
intrusion_with_gaps.to_csv(intrusion_out, index=False)

print(f"Salvo: {mitm_out} -> shape {mitm_with_gaps.shape}")
print(f"Salvo: {intrusion_out} -> shape {intrusion_with_gaps.shape}")

# Amostra final
print("\nAmostra MitM:")
print(mitm_with_gaps[['frame.time_epoch','mqtt.msgtype','publish_gap','connect_gap']].head())
print("\nAmostra Intrusion:")
print(intrusion_with_gaps[['frame.time_epoch','mqtt.msgtype','publish_gap','connect_gap']].head())


Salvo: ../data/processed/MitM_gap.csv -> shape (110668, 67)
Salvo: ../data/processed/intrusion_gap.csv -> shape (80893, 67)

Amostra MitM:
   frame.time_epoch  mqtt.msgtype  publish_gap  connect_gap
0      1.529668e+09           NaN          0.0          0.0
1      1.529668e+09           NaN          0.0          0.0
2      1.529668e+09           NaN          0.0          0.0
3      1.529668e+09           NaN          0.0          0.0
4      1.529668e+09           NaN          0.0          0.0

Amostra Intrusion:
   frame.time_epoch  mqtt.msgtype  publish_gap  connect_gap
0      1.535972e+09           NaN          0.0          0.0
1      1.535972e+09           NaN          0.0          0.0
2      1.535972e+09           NaN          0.0          0.0
3      1.535972e+09           NaN          0.0          0.0
4      1.535972e+09           NaN          0.0          0.0
